# Google Analytics Sample Dataset Marketing Exercise

This notebook is organized in the same order as the dashboard:

1. Examine what is in the data.
2. Check shape, types, and basic coverage.
3. Summarize channel, device, and journey patterns.
4. Ask only the business questions the data can support.

## Dataset Inputs

By default this notebook uses the local CSVs in `data/`.
If you have an uploaded export instead, replace the two paths below.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
SESSIONS_PATH = PROJECT_ROOT / 'data' / 'session_events.csv'
JOURNEYS_PATH = PROJECT_ROOT / 'data' / 'journey_summary.csv'

sessions = pd.read_csv(SESSIONS_PATH)
journeys = pd.read_csv(JOURNEYS_PATH)

sessions['session_date'] = pd.to_datetime(sessions['session_date'])
sessions['week_start'] = pd.to_datetime(sessions['week_start'])


## 1. What is in the data?

In [ ]:
summary = {
    'session_rows': len(sessions),
    'journey_rows': len(journeys),
    'date_min': sessions['session_date'].min(),
    'date_max': sessions['session_date'].max(),
    'channel_count': sessions['channel_group'].nunique(),
    'device_count': journeys['primary_device'].nunique(),
    'transactions': int(journeys['enrollment_equivalent_flag'].sum()),
    'revenue': float(journeys['transaction_revenue'].sum()),
}
pd.Series(summary)

In [ ]:
sessions.head()

In [ ]:
journeys.head()

## 2. Are the fields complete enough to use?

This is the fastest way to surface missingness before you build conclusions on top of a field.

In [ ]:
field_quality = pd.DataFrame({
    'table': ['sessions'] * len(sessions.columns) + ['journeys'] * len(journeys.columns),
    'field': list(sessions.columns) + list(journeys.columns),
    'missing_pct': list((sessions.isna().mean() * 100).round(2)) + list((journeys.isna().mean() * 100).round(2)),
})
field_quality.sort_values(['missing_pct', 'field'], ascending=[False, True]).head(20)

## 3. Create notebook-friendly marketing aliases

The local extract uses generic event names. This cell makes the analysis language explicit.

In [ ]:
sessions_renamed = sessions.rename(columns={
    'lead_event_flag': 'cart_intent_event_flag',
    'enrollment_equivalent_flag': 'transaction_flag',
})

journeys_renamed = journeys.rename(columns={
    'lead_flag': 'cart_intent_journey_flag',
    'enrollment_equivalent_flag': 'transaction_flag',
    'days_to_lead': 'days_to_cart_intent',
    'days_to_enrollment': 'days_to_transaction',
})

sessions_renamed[['session_id', 'channel_group', 'cart_intent_event_flag', 'transaction_flag']].head()

## 4. How do channel rankings change from traffic to transactions?

In [ ]:
channel_summary = (
    sessions_renamed.groupby('channel_group', as_index=False)
    .agg(
        sessions=('session_id', 'count'),
        cart_intents=('cart_intent_event_flag', 'sum'),
        transactions=('transaction_flag', 'sum'),
        revenue=('transaction_revenue', 'sum'),
        first_touch=('first_touch_credit', 'sum'),
        last_touch=('last_touch_credit', 'sum'),
        linear=('linear_credit', 'sum'),
    )
)

channel_summary['transactions_per_100_sessions'] = channel_summary['linear'] / channel_summary['sessions'] * 100
channel_summary['cart_to_transaction'] = channel_summary['linear'] / channel_summary['cart_intents'].replace(0, pd.NA)
channel_summary.sort_values('transactions_per_100_sessions', ascending=False)

In [ ]:
fig = px.scatter(
    channel_summary,
    x='sessions',
    y='transactions_per_100_sessions',
    size='cart_intents',
    color='channel_group',
    text='channel_group',
    title='Channel Efficiency: Sessions vs Attributed Transactions',
)
fig.update_traces(textposition='top center')
fig.show()

## 5. How much multi-touch behavior is in the dataset?

In [ ]:
touch_depth = (
    journeys_renamed.groupby('sessions_in_journey', as_index=False)
    .agg(
        journeys=('prospect_id', 'count'),
        transactions=('transaction_flag', 'sum'),
    )
)
touch_depth['transaction_rate'] = touch_depth['transactions'] / touch_depth['journeys']
touch_depth

In [ ]:
fig = go.Figure()
fig.add_bar(x=touch_depth['sessions_in_journey'], y=touch_depth['journeys'], name='Journeys')
fig.add_scatter(x=touch_depth['sessions_in_journey'], y=touch_depth['transaction_rate'] * 100, name='Transaction rate', yaxis='y2')
fig.update_layout(
    title='Touch Depth vs Transaction Rate',
    xaxis_title='Touches in journey',
    yaxis_title='Journeys',
    yaxis2=dict(title='Transaction rate (%)', overlaying='y', side='right'),
)
fig.show()

## 6. Which entry channels usually hand off to which closing channels?

In [ ]:
journey_flow = (
    journeys_renamed.loc[journeys_renamed['transaction_flag'] == 1]
    .groupby(['first_touch_channel', 'last_touch_channel'], as_index=False)
    .agg(transactions=('transaction_flag', 'sum'))
    .sort_values('transactions', ascending=False)
)
journey_flow.head(15)

## 7. Final question set for write-up

After the inspection work above, these are the questions worth answering in the final narrative:

- Which channels win on raw traffic, and which still win after attribution?
- Which channels act more like discoverers and which act more like closers?
- How much better do multi-touch journeys convert than one-touch journeys?
- Which paths are common enough to treat as repeatable patterns rather than anecdotes?
- Which channels deserve a follow-up experiment because their traffic share and transaction share do not match?